# Hydrological Analysis with AquaScope

This notebook demonstrates AquaScope's hydrology module — the bread-and-butter tools every water engineer needs.

**Contents:**
1. Flow Duration Curves (FDC)
2. Baseflow Separation
3. Recession Analysis
4. Flood Frequency Analysis
5. Low-Flow Statistics

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
%matplotlib inline

# Generate synthetic daily discharge (10 years)
rng = np.random.default_rng(42)
days = 365 * 10
dates = pd.date_range("2014-01-01", periods=days, freq="D")
t = np.arange(days) / 365.0
seasonal = 25 + 15 * np.sin(2 * np.pi * t)
noise = rng.exponential(5, days)
storms = rng.choice([0]*20 + [1], days) * rng.exponential(40, days)
Q = pd.Series(np.maximum(seasonal + noise + storms, 0.5), index=dates, name="Q")
print(f"Discharge: {Q.min():.1f}–{Q.max():.1f} m³/s, mean={Q.mean():.1f}")

## 1. Flow Duration Curve

In [ ]:
from aquascope.hydrology import flow_duration_curve
from aquascope.viz import plot_fdc

fdc = flow_duration_curve(Q)
print("Key percentiles:")
for p in [5, 10, 50, 90, 95]:
    print(f"  Q{p} = {fdc.percentiles[p]:.2f} m³/s")

plot_fdc(Q, title="Flow Duration Curve (10-yr synthetic)")

## 2. Baseflow Separation

Two methods available:
- **Lyne-Hollick** (1979): Single-parameter, most widely used
- **Eckhardt** (2005): Two-parameter, accounts for aquifer type

In [ ]:
from aquascope.hydrology import lyne_hollick, eckhardt
from aquascope.viz.hydro import plot_hydrograph

bf_lh = lyne_hollick(Q)
bf_ek = eckhardt(Q, bfi_max=0.80)
print(f"Lyne-Hollick BFI: {bf_lh.bfi:.3f}")
print(f"Eckhardt BFI:     {bf_ek.bfi:.3f}")

# Plot first year
yr1 = bf_lh.df.iloc[:365].rename(columns={"total": "discharge"})
plot_hydrograph(yr1, title="Hydrograph with Baseflow (Year 1)")

## 3. Recession Analysis

In [ ]:
from aquascope.hydrology import recession_analysis

rec = recession_analysis(Q)
print(f"Recession segments: {len(rec.segments)}")
print(f"Recession constant: {rec.recession_constant:.1f} days")
print(f"Half-life:          {rec.half_life_days:.1f} days")
print(f"R²:                 {rec.r_squared:.4f}")

## 4. Flood Frequency Analysis

In [ ]:
from aquascope.hydrology import fit_gev, fit_lp3
from aquascope.viz import plot_return_periods

gev = fit_gev(Q)
lp3 = fit_lp3(Q)

print("GEV Return Periods:")
for rp in [2, 10, 50, 100]:
    ci = gev.confidence_intervals.get(rp)
    ci_str = f"  [{ci[0]:.0f}, {ci[1]:.0f}]" if ci else ""
    print(f"  {rp:>3d}-yr: {gev.return_periods[rp]:.1f}{ci_str}")

plot_return_periods(gev.return_periods, observed_max=float(Q.max()),
                    title="Flood Frequency (GEV)")

## 5. Low-Flow Statistics

In [ ]:
from aquascope.hydrology import low_flow_stat

q7_10 = low_flow_stat(Q, n_day=7, return_period=10)
q30_5 = low_flow_stat(Q, n_day=30, return_period=5)
print(f"7Q10 = {q7_10:.2f} m³/s")
print(f"30Q5 = {q30_5:.2f} m³/s")

## Summary

All analysis results are shown above. These tools are also available from the CLI:
```bash
aquascope hydro --analysis fdc --file discharge.csv
aquascope hydro --analysis baseflow --file discharge.csv
aquascope hydro --analysis flood-freq --file discharge.csv
```